# Villarica_Spark
Análisis de ventas de la cafetería **Villa Rica** con PySpark, a partir del CSV exportado del modelo estrella (`FactVenta`).

## Paso 0. Parámetros
Cambia `RUTA_CSV` por la ruta donde subiste el archivo (en Databricks: un Volume, p. ej. `/Volumes/workspace/bd_villarica/grupo_01/RESULTADOS-Villarica.csv`).

In [ ]:
RUTA_CSV = "/Volumes/workspace/bd_villarica/grupo_01/RESULTADOS-Villarica.csv"
NOMBRE_BD = "bd_villarica"

## Paso 1. Crear la sesión y cargar el CSV

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, to_date, sum as spark_sum, round as spark_round

spark = SparkSession.builder.appName("CafeteriaVillarica").getOrCreate()

In [ ]:
df = spark.read.csv(RUTA_CSV, header=False, sep=";", inferSchema=True)

df = df.toDF("id_detalle", "id_venta", "fecha", "producto", "categoria", "cantidad", "importe", "metodo_pago")

## Paso 2. Revisar la estructura de los datos

In [ ]:
df.printSchema()
df.describe().show()
print("Total de registros:", df.count())

## Paso 3. Eliminar espacios innecesarios

In [ ]:
df = (df.withColumn("producto", trim(col("producto")))
        .withColumn("categoria", trim(col("categoria")))
        .withColumn("metodo_pago", trim(col("metodo_pago"))))
df.show(5, truncate=False)

## Paso 4. Identificar y tratar valores nulos

In [ ]:
df.select([col(c).isNull().cast("int").alias(c) for c in df.columns]).groupBy().sum().show()

df = df.na.drop(subset=df.columns)

## Paso 5. Corregir tipos de datos

In [ ]:
df = (df.withColumn("id_venta", col("id_venta").cast("integer"))
        .withColumn("fecha", to_date(col("fecha"), "yyyy-MM-dd"))
        .withColumn("cantidad", col("cantidad").cast("integer"))
        .withColumn("importe", col("importe").cast("decimal(12,2)")))
df.printSchema()

## Paso 6. Filtrar registros incorrectos

In [ ]:
df = df.filter(
    (col("id_detalle") > 0) &
    (col("id_venta") > 0) &
    (col("fecha").isNotNull()) &
    (col("producto") != "") &
    (col("categoria") != "") &
    (col("cantidad") > 0) &
    (col("importe") > 0) &
    (col("metodo_pago") != "")
)
df.show(10, truncate=False)

## Paso 7. Eliminar registros duplicados
Una venta puede tener el mismo producto en dos líneas (mismo ticket, producto, cantidad e importe). Por eso los duplicados se detectan por **`id_detalle`** (la línea de venta) y no por todas las columnas.

In [ ]:
df = df.dropDuplicates(["id_detalle"])
print("Registros después de eliminar duplicados:", df.count())

## Paso 8. Crear columnas calculadas
El CSV trae el **importe de la línea** (cantidad × precio). El precio unitario se obtiene dividiendo.

In [ ]:
df = df.withColumn("precio_unitario", spark_round(col("importe") / col("cantidad"), 2))

df.select(
    spark_sum("cantidad").alias("Total_Unidades"),
    spark_sum("importe").alias("Total_Ventas")
).show()

## Paso 9. Seleccionar columnas necesarias

In [ ]:
df = df.select("id_detalle", "id_venta", "fecha", "producto", "categoria",
               "cantidad", "precio_unitario", "importe", "metodo_pago")
df.show(10, truncate=False)

## Paso 10. Agrupar y analizar las ventas

Ventas por categoría:

In [ ]:
df_ventas_categoria = (df.groupBy("categoria")
                         .agg(spark_sum("cantidad").alias("unidades"), spark_sum("importe").alias("importe"))
                         .orderBy(col("importe").desc()))
df_ventas_categoria.show(truncate=False)

Ventas por método de pago:

In [ ]:
df_ventas_pago = (df.groupBy("metodo_pago")
                    .agg(spark_sum("importe").alias("importe"))
                    .orderBy(col("importe").desc()))
df_ventas_pago.show(truncate=False)

Filtrar productos de la categoría Cafés:

In [ ]:
df_cafes = df.filter(col("categoria") == "Cafés").select("producto", "cantidad", "precio_unitario", "importe")
df_cafes.show(10, truncate=False)

Top 10 productos por importe:

In [ ]:
(df.groupBy("producto")
   .agg(spark_sum("cantidad").alias("unidades"), spark_sum("importe").alias("importe"))
   .orderBy(col("importe").desc())
   .show(10, truncate=False))

Ventas por día:

In [ ]:
(df.groupBy("fecha")
   .agg(spark_sum("importe").alias("importe"))
   .orderBy("fecha")
   .show(20, truncate=False))

## Paso 11. Crear la base de datos y guardar la tabla

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {NOMBRE_BD}")

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{NOMBRE_BD}.tb_ventas_villarica")

In [ ]:
dfVentasGuardadas = spark.table(f"{NOMBRE_BD}.tb_ventas_villarica")
display(dfVentasGuardadas)   # en Databricks; fuera de Databricks usa .show()

## Gráficos
En Databricks: ejecuta cada celda y en el resultado pulsa **+ → Visualization** (barras para categorías y productos, pastel para método de pago).

In [ ]:
display(df.groupBy("categoria").sum("importe"))

In [ ]:
display(df.groupBy("producto").sum("cantidad").orderBy(col("sum(cantidad)").desc()).limit(10))

In [ ]:
display(df.groupBy("metodo_pago").sum("importe"))